# BlendColab / GodotBakeCloud — Distributed Free Cloud GPU Farm

Render **Blender Cycles** animations (`OptiX` on T4 / `CUDA` on P100) or bake **Godot 4 `LightmapGI`** scenes headlessly over Vulkan + Xvfb using Google Drive as a shared distributed lock queue.

### Multi-GPU Scaling Tip
- Open **2 to 4 browser tabs** (Google Colab + Kaggle `T4 x2`) pointed at the same `DRIVE_WORKSPACE` folder.
- Workers automatically claim unlocked frames/scenes, send heartbeats every 30s, recover orphaned locks if a tab disconnects, and push preview thumbnails to Discord.

In [3]:
#@title 1. Mount Google Drive & Bootstrap Blender / Godot Binaries
import json
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/DeviouslyCalamitous/BlendColab-GodotBakeCloud.git" #@param {type:"string"}
DRIVE_WORKSPACE = "BlendColab_Farm" #@param {type:"string"}
INSTALL_ENGINES = "both" #@param ["both", "blender", "godot"]

# Mount Google Drive directly in Colab's main cell so the permission popup works
if Path("/content").exists():
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

base_dir = Path("/content") if Path("/content").exists() else Path("/kaggle/working")
repo_dir = base_dir / "BlendColab-GodotBakeCloud"

if not repo_dir.exists():
    print(f"Cloning {REPO_URL}...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=False)

os.chdir(repo_dir)

# Run with !python3 so live download progress & any errors print directly on screen
!python3 scripts/env_bootstrap.py --engine "{INSTALL_ENGINES}" --workspace-name "{DRIVE_WORKSPACE}" --repo-dir "{repo_dir}"

assert Path("/tmp/cloud_farm_paths.json").exists(), "Setup failed — check the error message printed above."

with open("/tmp/cloud_farm_paths.json", "r", encoding="utf-8") as f:
    PATHS = json.load(f)

print("\nDetected GPU Hardware:")
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv,noheader
print(f"\nShared Drive Workspace Ready: {PATHS['workspace']}")
print("Upload your .blend file or Godot 4 project folder into:", f"{PATHS['workspace']}/projects/")

Mounted at /content/drive
20:05:07 | INFO    | Downloading https://download.blender.org/release/Blender4.2/blender-4.2.4-linux-x64.tar.xz...
  -> 336.7 MB / 336.7 MB (100%)
20:05:09 | INFO    | Caching archive to Google Drive: /content/drive/MyDrive/BlendColab_Farm/cache/blender-4.2.4-linux-x64.tar.xz
20:05:40 | INFO    | Extracted blender in 29.7s -> /content/cloud_farm_tools/blender/blender-4.2.4-linux-x64/blender
20:05:40 | INFO    | Downloading https://github.com/godotengine/godot/releases/download/4.3-stable/Godot_v4.3-stable_linux.x86_64.zip...
  -> 48.0 MB / 47.9 MB (100%)
20:05:41 | INFO    | Caching archive to Google Drive: /content/drive/MyDrive/BlendColab_Farm/cache/Godot_v4.3-stable_linux.x86_64.zip
20:05:43 | INFO    | Extracted godot in 1.3s -> /content/cloud_farm_tools/godot/Godot_v4.3-stable_linux.x86_64
20:05:43 | INFO    | Bootstrap complete. Paths saved to /tmp/cloud_farm_paths.json:
{
  "platform": "colab",
  "workspace": "/content/drive/MyDrive/BlendColab_Farm",
  

In [ ]:
#@title 2. Configure & Start Distributed Render / Bake Worker
import json
import subprocess
from pathlib import Path

with open("/tmp/cloud_farm_paths.json", "r", encoding="utf-8") as f:
    PATHS = json.load(f)

#@markdown ### Engine & Project Selection
ENGINE = "blender" #@param ["blender", "godot"]
PROJECT_SUBPATH = "projects/scene.blend" #@param {type:"string"}

#@markdown ### Blender Cycles Settings (Ignored in Godot Mode)
START_FRAME = 1 #@param {type:"integer"}
END_FRAME = 120 #@param {type:"integer"}
SAMPLES_OVERRIDE = 0 #@param {type:"integer"}
ASSEMBLE_MP4 = True #@param {type:"boolean"}
FPS = 24 #@param {type:"integer"}
POOL_MULTI_GPU = False #@param {type:"boolean"}

#@markdown ### Godot 4 LightmapGI Settings (Ignored in Blender Mode)
GODOT_SCENES = "res://main.tscn" #@param {type:"string"}

#@markdown ### Telemetry & Quota Protection
DISCORD_WEBHOOK_URL = "" #@param {type:"string"}
NOTIFY_EVERY_N_TASKS = 5 #@param {type:"integer"}
AUTO_DISCONNECT_WHEN_DONE = False #@param {type:"boolean"}

workspace = Path(PATHS["workspace"])
project_path = workspace / PROJECT_SUBPATH
assert project_path.exists(), f"Project not found on Drive: {project_path}"

cmd = [
    "python3", "scripts/render_worker.py",
    "--engine", ENGINE,
    "--workspace", str(workspace),
    "--notify-every", str(max(1, NOTIFY_EVERY_N_TASKS)),
]

if DISCORD_WEBHOOK_URL.strip():
    cmd.extend(["--discord-webhook", DISCORD_WEBHOOK_URL.strip()])
if AUTO_DISCONNECT_WHEN_DONE:
    cmd.append("--auto-disconnect")
if POOL_MULTI_GPU:
    cmd.append("--pool-gpus")

if ENGINE == "blender":
    cmd.extend([
        "--blender-bin", PATHS["blender_bin"],
        "--blend-file", str(project_path),
        "--start-frame", str(START_FRAME),
        "--end-frame", str(END_FRAME),
        "--fps", str(FPS),
    ])
    if SAMPLES_OVERRIDE > 0:
        cmd.extend(["--samples", str(SAMPLES_OVERRIDE)])
    if ASSEMBLE_MP4:
        cmd.append("--assemble-video")
else:
    # Automatically inject and enable godot_bake_cloud EditorPlugin into project
    from scripts.env_bootstrap import inject_godot_addon
    inject_godot_addon(Path.cwd(), project_path)
    cmd.extend([
        "--godot-bin", PATHS["godot_bin"],
        "--godot-project", str(project_path),
        "--godot-scenes", GODOT_SCENES,
    ])

print("Launching worker command:\n ", " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
#@title 3. Queue Inspector & Lock Maintenance Utility
import json
import time
from pathlib import Path

with open("/tmp/cloud_farm_paths.json", "r", encoding="utf-8") as f:
    PATHS = json.load(f)

FORCE_CLEAR_ALL_LOCKS = False #@param {type:"boolean"}
RESET_DONE_MARKERS = False #@param {type:"boolean"}

queue_dir = Path(PATHS["workspace"]) / "render_queue"
done_files = sorted(queue_dir.glob("*.done"))
lock_files = sorted(queue_dir.glob("*.lock"))

print(f"Completed Tasks : {len(done_files)}")
print(f"Active Locks    : {len(lock_files)}")

for lf in lock_files:
    try:
        info = json.loads(lf.read_text(encoding="utf-8"))
        age = time.time() - info.get("heartbeat_at", 0)
        print(f"  - {lf.name}: worker={info.get('worker_id')} | gpu={info.get('gpu_name')} | heartbeat_age={age:.0f}s")
    except Exception:
        print(f"  - {lf.name}: unreadable lockfile")

if FORCE_CLEAR_ALL_LOCKS:
    for lf in lock_files:
        lf.unlink(missing_ok=True)
    print("Cleared all .lock files.")

if RESET_DONE_MARKERS:
    for df in done_files:
        df.unlink(missing_ok=True)
    print("Cleared all .done markers (queue reset for full re-render).")